# Gaming Industry Demand Prediction Using a Neural Network

**Subject:** Neural Networks and Deep Learning  
**Project:** Predicting demand for video games

## Aim
To build a simple deep learning model that predicts the expected demand for a video game.

## Objectives
1. Create and understand a small gaming-industry dataset.
2. Prepare the data for training a neural network.
3. Train an Artificial Neural Network (ANN) to predict game demand.
4. Evaluate the model using MAE, RMSE and R² score.

## Theory
A neural network learns patterns from input data through connected layers of artificial neurons. In this project, the model learns how factors such as game genre, platform, price, marketing budget, pre-orders and ratings relate to demand. It is a regression task because the output is a number of expected units.

> **Note:** This notebook creates a synthetic example dataset. It is suitable for learning and demonstrating the model, but it is not actual market-sales data.

## 1. Import Required Libraries
We use Pandas and NumPy for data handling, Matplotlib for plots, Scikit-learn for preprocessing and evaluation, and TensorFlow/Keras to build the neural network.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

# Keep results reproducible
np.random.seed(42)
tf.random.set_seed(42)

## 2. Create the Gaming Industry Dataset
The dataset contains game information and market signals. The target column, `Demand_Units`, represents the simulated number of units demanded.

In [ ]:
n = 1000

genres = np.random.choice(['Action', 'Sports', 'RPG', 'Simulation', 'Puzzle'], n)
platforms = np.random.choice(['PC', 'PlayStation', 'Xbox', 'Mobile'], n)
months = np.random.choice(
    ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], n
)

price = np.random.uniform(5, 80, n).round(2)
marketing_budget = np.random.uniform(10, 150, n).round(1)  # thousand USD
social_buzz = np.random.randint(0, 101, n)
pre_orders = np.random.randint(100, 10001, n)
user_rating = np.random.uniform(2.5, 5.0, n).round(1)
discount = np.random.randint(0, 51, n)

# Simple effects used to create realistic-looking example demand values
genre_effect = {'Action': 1300, 'Sports': 900, 'RPG': 1100, 'Simulation': 500, 'Puzzle': 300}
platform_effect = {'PC': 200, 'PlayStation': 600, 'Xbox': 300, 'Mobile': 400}
month_effect = {
    'Jan': 0, 'Feb': -100, 'Mar': 100, 'Apr': 0, 'May': 100, 'Jun': 250,
    'Jul': 150, 'Aug': 100, 'Sep': 200, 'Oct': 700, 'Nov': 1100, 'Dec': 1300
}

base_demand = (
    900
    + 0.85 * pre_orders
    + 18 * marketing_budget
    + 15 * social_buzz
    + 500 * (user_rating - 3.5)
    - 18 * price
    + 20 * discount
    + np.array([genre_effect[x] for x in genres])
    + np.array([platform_effect[x] for x in platforms])
    + np.array([month_effect[x] for x in months])
)

demand = np.maximum(100, base_demand + np.random.normal(0, 450, n)).astype(int)

data = pd.DataFrame({
    'Genre': genres,
    'Platform': platforms,
    'Month': months,
    'Price_USD': price,
    'Marketing_Budget_Thousand_USD': marketing_budget,
    'Social_Media_Buzz': social_buzz,
    'Pre_Orders': pre_orders,
    'User_Rating': user_rating,
    'Discount_Percent': discount,
    'Demand_Units': demand
})

# Display the first five records
data.head()

## 3. Explore the Dataset
Check the dataset size, column types, and average demand.

In [ ]:
print('Dataset shape:', data.shape)
print('\nColumn information:')
data.info()
print('\nAverage demand:', round(data['Demand_Units'].mean(), 0), 'units')

# View basic statistics
data.describe(numeric_only=True)

## 4. Prepare the Data
Convert text categories into numeric columns using one-hot encoding. Split the data into training and testing sets, then scale the inputs.

In [ ]:
X = data.drop('Demand_Units', axis=1)
y = data['Demand_Units']

# Convert Genre, Platform and Month into numeric indicator columns
X = pd.get_dummies(X, columns=['Genre', 'Platform', 'Month'], dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Fit scaling only on the training data to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Training records:', X_train.shape[0])
print('Testing records:', X_test.shape[0])
print('Number of input features:', X_train.shape[1])

## 5. Build the Neural Network Model
The model has an input layer, two hidden layers with ReLU activation, and one output neuron to predict demand.

In [ ]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)  # One numeric output: predicted demand
])

model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mean_absolute_error']
)

model.summary()

## 6. Train the Model
Train the network for 50 epochs. A validation split is used to monitor performance during training.

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.20,
    verbose=1
)

## 7. Plot Training and Validation Loss
A decreasing loss generally indicates that the model is learning from the data.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Model Training Performance')
plt.legend()
plt.show()

## 8. Evaluate the Model
Use the testing data to calculate MAE, RMSE and R² score. Lower MAE/RMSE and an R² closer to 1 indicate better predictions.

In [ ]:
predictions = model.predict(X_test_scaled).flatten()

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print('Mean Absolute Error (MAE):', round(mae, 2), 'units')
print('Root Mean Squared Error (RMSE):', round(rmse, 2), 'units')
print('R² Score:', round(r2, 3))

## 9. Compare Actual and Predicted Demand
View a few predictions from the test set.

In [ ]:
comparison = pd.DataFrame({
    'Actual_Demand': y_test.values,
    'Predicted_Demand': np.round(predictions).astype(int)
})
comparison.head(10)

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, predictions, alpha=0.6)
plt.xlabel('Actual Demand (units)')
plt.ylabel('Predicted Demand (units)')
plt.title('Actual vs Predicted Game Demand')
plt.show()

## 10. Predict Demand for a New Game
Enter sample information for a new game. The trained model will estimate its demand.

In [ ]:
new_game = pd.DataFrame([{
    'Genre': 'Action',
    'Platform': 'PlayStation',
    'Month': 'Nov',
    'Price_USD': 49.99,
    'Marketing_Budget_Thousand_USD': 100,
    'Social_Media_Buzz': 85,
    'Pre_Orders': 7000,
    'User_Rating': 4.5,
    'Discount_Percent': 10
}])

# Apply the same one-hot columns and scaling used during training
new_game_encoded = pd.get_dummies(new_game, columns=['Genre', 'Platform', 'Month'], dtype=int)
new_game_encoded = new_game_encoded.reindex(columns=X.columns, fill_value=0)
new_game_scaled = scaler.transform(new_game_encoded)

predicted_demand = model.predict(new_game_scaled)[0][0]
print('Estimated demand for the new game:', round(max(0, predicted_demand)), 'units')

## Outcome
A simple Artificial Neural Network was created and trained to estimate video game demand. The model was evaluated using MAE, RMSE and R² score, and was also used to predict demand for a new game.

## Conclusion
The project demonstrates how deep learning can be applied to the gaming industry. Factors such as genre, platform, price, marketing, pre-orders, ratings and launch month can help estimate demand. The example uses generated data, so real business decisions would require a reliable dataset of actual game sales and market activity.